1. Import Libraries and Load Environment Variables

- firebase_admin: Firebase SDK for Python to interact with Firebase services (e.g., Firestore, Realtime Database, and Storage).
- dotenv: Loads environment variables securely from a .env file, which is used to store sensitive credentials.
- pandas: Used for reading and processing the dataset.
- os: Handles file paths and environment variable operations.

In [170]:
import firebase_admin
from firebase_admin import credentials, storage
from firebase_admin import db
import pandas as pd
import os
import dotenv
dotenv.load_dotenv()

True

# Firebase Init

a. Prepare Service Account Info

- Reads Firebase service account credentials from environment variables.
- Formats the private_key properly, as it might include escaped newlines (\\n) in .env.

In [172]:
service_account_info = {
    "type": os.getenv("FIREBASE_TYPE"),
    "project_id": os.getenv("FIREBASE_PROJECT_ID"),
    "private_key_id": os.getenv("FIREBASE_PRIVATE_KEY_ID"),
    "private_key": os.getenv("FIREBASE_PRIVATE_KEY").replace("\\n", "\n"),
    "client_email": os.getenv("FIREBASE_CLIENT_EMAIL"),
    "client_id": os.getenv("FIREBASE_CLIENT_ID"),
    "auth_uri": os.getenv("FIREBASE_AUTH_URI"),
    "token_uri": os.getenv("FIREBASE_TOKEN_URI"),
    "auth_provider_x509_cert_url": os.getenv("FIREBASE_AUTH_PROVIDER_X509_CERT_URL"),
    "client_x509_cert_url": os.getenv("FIREBASE_CLIENT_X509_CERT_URL"),
    "universe_domain": os.getenv("FIREBASE_UNIVERSE_DOMAIN")
}
  

b. Reinitialize Firebase App (if needed)

Ensures that a Firebase app is not already initialized. If it is, deletes the existing app.
Initializes Firebase using the provided credentials:
- Storage Bucket: Configured for storing product images.
- Realtime Database URL: Configured for storing product details.

In [173]:
# Check and re-initialize Firebase app if needed
if firebase_admin._apps:
    firebase_admin.delete_app(firebase_admin.get_app())

cred = credentials.Certificate(service_account_info)
firebase_admin.initialize_app(cred, {
    "storageBucket": "coffeeshop-app-20d1a.firebasestorage.app",
    "databaseURL": "https://coffeeshop-app-20d1a-default-rtdb.firebaseio.com",
})



Creates a reference to:
- Firebase Storage Bucket: For uploading images.
- Realtime Database: For uploading product details.

In [174]:
bucket = storage.bucket()
products_collection = db.reference("products")


# Upload Data

- Reads the product data from a JSON Lines file (.jsonl) into a Pandas DataFrame.
- Preview the first two rows to verify the structure of the data.

In [175]:
image_folder_path = './products/images/'

In [176]:
products_collection = db.reference('products')

In [177]:
df = pd.read_json('products/products.jsonl',lines=True)
df.head(2)

,name,category,description,ingredients,price,rating,image_path
0,Cappuccino,Coffee,A rich and creamy cappuccino made with freshly...,"[Espresso, Steamed Milk, Milk Foam]",4.50,4.7,cappuccino.jpg
1,Jumbo Savory Scone,Bakery,"Deliciously flaky and buttery, this jumbo savo...","[Flour, Butter, Cheese, Herbs, Baking Powder, ...",3.25,4.3,SavoryScone.webp


4. Define Image Upload Function

In [178]:
def upload_image(bucket, image_path):
    image_name = image_path.split('/')[-1]
    blob = bucket.blob(f'product_images/{image_name}')
    # Upload image
    blob.upload_from_filename(image_path)
    # Make the image publicly accessible and get its URL
    blob.make_public()
    return blob.public_url


Inputs:
- bucket: Firebase Storage bucket object.
- image_path: Local path to the image file.

Steps:
- Creates a reference (blob) in Firebase Storage under the product_images/ folder.
- Uploads the image to Firebase Storage using upload_from_filename.
- Makes the image publicly accessible using make_public and retrieves the public URL.

In [179]:
for index, row in df.iterrows():
    print(index, row['name'])
    
    image_path = os.path.join(image_folder_path,row['image_path'])
    
    image_url = upload_image(bucket,image_path)
    product_data = row.to_dict()
    product_data.pop('image_path')
    product_data['image_url']= image_url
    
    # Add to Firestore
    products_collection.push().set(product_data)

0 Cappuccino
1 Jumbo Savory Scone
2 Latte
3 Chocolate Chip Biscotti
4 Espresso shot
5 Hazelnut Biscotti
6 Chocolate Croissant
7 Dark chocolate
8 Cranberry Scone
9 Croissant
10 Almond Croissant
11 Ginger Biscotti
12 Oatmeal Scone
13 Ginger Scone
14 Chocolate syrup
15 Hazelnut syrup
16 Carmel syrup
17 Sugar Free Vanilla syrup


5. Process and Upload Data
a. Iterate Over Each Product

- Loops through each row in the DataFrame, representing a product.
- Prints the product index and name for progress tracking.


b. Upload Images

- Constructs the full local path for each product's image.
- Calls the upload_image function to upload the image to Firebase Storage and retrieve its public URL.

c. Prepare Product Data

- Converts the row into a dictionary for easy manipulation.
- Removes the image_path field (not needed in Firebase) and replaces it with the image_url.

d. Upload Product Data to Realtime Database

- Pushes the processed product data (including the image URL) to the Realtime Database under the products node.